# Real-world Data Wrangling
### Fullname: Gia Huy _ Thuong _ Nhat
### ID: 
### Class: 22JIT

## 1. Gather data


### **1.1.** Problem Statement

This project aims to analyze high school graduation exam (THPTQG) scores across multiple years (2020-2025) to understand trends in score distribution and competition levels.

In this dataset, I would like to answer the following research questions:

Question 1: Tốc độ tăng trưởng số lượng thí sinh dự thi năm 2025 so với giai đoạn 2020-2024 thay đổi ra sao, và sự gia tăng khối lượng thí sinh này tạo ra áp lực cạnh tranh (tỷ lệ chọi) như thế nào lên các nhóm trường Đại học top đầu?

Question 2: Có sự dịch chuyển bất thường nào trong phổ điểm của các môn học từng có hiện tượng "tăng vọt" về điểm số trong năm 2024 (như Lịch sử, Địa lý, Ngữ Văn) hay không? Đề thi năm 2025 đã đưa phổ điểm các môn này về mức cân bằng hay tiếp tục lạm phát điểm?

Question 3: Ngưỡng điểm để lọt vào "Top 5% và Top 10% thí sinh xuất sắc nhất" ở các khối thi truyền thống (A, A1, B, C, D) năm 2025 dịch chuyển bao nhiêu điểm so với 2024?

Question 4: Tại các vùng điểm nóng cạnh tranh (từ 24 đến 28 điểm), số lượng thí sinh tích lũy dư thừa hay thiếu hụt bao nhiêu người so với năm 2024 ở từng khối thi; kết hợp với biến số về chỉ tiêu và phương thức xét tuyển, điểm chuẩn của các trường Đại học top đầu (như Bách Khoa, Ngoại Thương, Kinh Tế Quốc Dân,...) sẽ biến động theo xu hướng nào?

Question 5: Mối tương quan điểm số giữa các môn học (Ví dụ: Toán và Ngoại ngữ, hoặc Ngữ Văn và Lịch Sử) năm 2025 có sự thay đổi nào không? Những thí sinh học giỏi môn Tự nhiên có xu hướng đạt điểm cao môn Ngoại ngữ như các năm trước hay không?

Question 6: Bản đồ phân bổ trung vị điểm và "độ lệch chuẩn" năm 2025 cho thấy khoảng cách về năng lực học tập và chất lượng giáo dục giữa các thành phố lớn (Hà Nội, TP.HCM) với các khu vực miền núi/vùng ven đang thu hẹp hay ngày càng giãn rộng?


### **1.2.** Gather at least two datasets using two different data gathering methods


Gather each of the 2 pieces of data from different sources:


In [87]:
# import libraries:
import pandas as pd
import numpy as np
import os

In [88]:
data_files = {
    '2021': 'dataset/diem_thi_2021.csv',
    '2022': 'dataset/diem_thi_2022.csv',
    '2023': 'dataset/diem_thi_2023.csv',
    '2024': 'dataset/diem_thi_2024.csv',
    '2025_ctcu': 'dataset/diem_thi_2025_ctcu.csv',
    '2025_moi_1': 'dataset/diem_thi_2025_moi_1.csv',
    '2025_moi_2': 'dataset/diem_thi_2025_moi_2.csv'
}

dfs = {}
for year, file_path in data_files.items():
    # Try different encodings, focusing on utf-8-sig for 2025 files
    try:
        dfs[year] = pd.read_csv(file_path, encoding='utf-8-sig')
    except:
        try:
            dfs[year] = pd.read_csv(file_path, encoding='utf-8')
        except:
            dfs[year] = pd.read_csv(file_path, encoding='latin1')
    print(f"Loaded {year}: {dfs[year].shape[0]} records")

C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_23904\3315485516.py:15: DtypeWarning: Columns (0: Tên, 1: Ngày Sinh, 2: Giới tính) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[year] = pd.read_csv(file_path, encoding='utf-8-sig')


Loaded 2021: 1857877 records
Loaded 2022: 995441 records
Loaded 2023: 1022060 records
Loaded 2024: 1061605 records
Loaded 2025_ctcu: 22090 records
Loaded 2025_moi_1: 1000000 records
Loaded 2025_moi_2: 131136 records


In [89]:
# Write data to local file 


## 2. Assess data


I will assess the data both visually and programmatically to identify any data quality(content) issues and tidiness(structual) issues.


### Issue 1: Inconsistent Column Names and Encoding Issues

In [90]:
# Inspecting column names across years
for year, df in dfs.items():
    print(f'{year}: {df.columns.tolist()[:5]}...')

2021: ['SBD', 'Tên', 'Ngày Sinh', 'Giới tính', 'Toán']...
2022: ['sbd', 'toan', 'ngu_van', 'ngoai_ngu', 'vat_li']...
2023: ['sbd', 'toan', 'ngu_van', 'ngoai_ngu', 'vat_li']...
2024: ['sbd', 'toan', 'ngu_van', 'ngoai_ngu', 'vat_li']...
2025_ctcu: ['STT', 'SOBAODANH', 'Toán', 'Văn', 'Lí']...
2025_moi_1: ['STT', 'SOBAODANH', 'Toán', 'Văn', 'Lí']...
2025_moi_2: ['STT', 'SOBAODANH', 'Toán', 'Văn', 'Lí']...


In [91]:
# Checking column name consistency
all_cols = set()
for df in dfs.values():
    all_cols.update(df.columns)
print(f'Total unique raw columns: {len(all_cols)}')

Total unique raw columns: 40


#### Observations:
Many columns represent the same subject but have different names (e.g., 'Toán', 'ToAn', 'toan'). Some years (2021, 2025) have mangled Vietnamese characters due to encoding differences (UTF-8 vs Latin1).

### Issue 2: Incorrect Data Types for Student ID (SBD)

In [92]:
# Checking the format of SBD
for year, df in dfs.items():
    print(f'{year} SBD example:', df.iloc[:, 0].head(1).values)

2021 SBD example: [18014547]
2022 SBD example: [1000001]
2023 SBD example: [1000001]
2024 SBD example: [1000001]
2025_ctcu SBD example: [1]
2025_moi_1 SBD example: [1]
2025_moi_2 SBD example: [1000001]


In [93]:
# Checking data types of SBD columns
for year, df in dfs.items():
    sbd_col = [c for c in df.columns if 'sbd' in c.lower() or 'baodanh' in c.lower()][0]
    print(f'{year} SBD type: {df[sbd_col].dtype}')

2021 SBD type: int64
2022 SBD type: int64
2023 SBD type: int64
2024 SBD type: int64
2025_ctcu SBD type: int64
2025_moi_1 SBD type: int64
2025_moi_2 SBD type: int64


#### Observations:
Student IDs (SBD) are often read as integers or floats. This causes loss of leading zeros (e.g., '01000001' becomes '1000001'), which is critical for identifying provinces accurately.

### Issue 3: Structural Differences between Old and New Curricula

In [94]:
# Comparing 2024 (Old) vs 2025 (New)
print('2024:', [c for c in dfs['2024'].columns if 'ma' in c.lower()])
print('2025 Moi:', [c for c in dfs['2025_moi_1'].columns if 'tin' in c.lower() or 'ktpl' in c.lower()])

2024: ['ma_ngoai_ngu']
2025 Moi: ['Tin học']


In [95]:
# Checking for unique subjects in 2025
new_subjects = ['tin_hoc', 'gd_ktpl', 'cong_nghe']
for year in ['2024', '2025_moi_1']:
    print(f'{year} subjects:', [c for c in dfs[year].columns if any(s in c.lower() for s in new_subjects)])

2024 subjects: []
2025_moi_1 subjects: []


#### Observations:
The 2025 graduates are split into two groups: those following the old curriculum and those following the new one. The new curriculum introduced subjects like 'Tin học' and 'GD Kinh tế và Pháp luật', which need to be accounted for separately during merging.

### Issue 4: Presence of Irrelevant Metadata (STT, Year, Province)

In [96]:
# Checking for non-score columns
display(dfs['2021'].head(2))

,SBD,Tên,Ngày Sinh,Giới tính,Toán,Văn,Lý,Hoá,Sinh,Lịch Sử,Địa Lý,GDCD,Ngoại Ngữ,Year,code,province
0,18014547,NaN,NaN,NaN,6.4,6.75,NaN,NaN,NaN,4.75,7.00,6.50,4.2,2020,18,Bắc Giang
1,18014530,NaN,NaN,NaN,7.6,6.00,NaN,NaN,NaN,3.75,7.75,7.75,2.8,2020,18,Bắc Giang


In [97]:
# Listing candidate columns to drop
drop_candidates = ['stt', 'tên', 'ngày_sinh', 'giới_tính', 'code', 'province']
for year, df in dfs.items():
    found = [c for c in df.columns if any(d in c.lower() for d in drop_candidates)]
    if found: print(f'{year} has extra columns: {found}')

2021 has extra columns: ['Tên', 'code', 'province']
2025_ctcu has extra columns: ['STT']
2025_moi_1 has extra columns: ['STT']
2025_moi_2 has extra columns: ['STT']


#### Observations:
Some datasets include 'STT' (Index), 'Tên' (Name), or 'Province' names. Since we are doing a cross-year score analysis, these personal details or redundant indexes should be removed to protect privacy and maintain a lean dataset.

In [98]:
# Visual inspection
for year, df in dfs.items():
    print(f"\n--- {year} ---")
    display(df.head(2))
    print(df.info())


--- 2021 ---


,SBD,Tên,Ngày Sinh,Giới tính,Toán,Văn,Lý,Hoá,Sinh,Lịch Sử,Địa Lý,GDCD,Ngoại Ngữ,Year,code,province
0,18014547,NaN,NaN,NaN,6.4,6.75,NaN,NaN,NaN,4.75,7.00,6.50,4.2,2020,18,Bắc Giang
1,18014530,NaN,NaN,NaN,7.6,6.00,NaN,NaN,NaN,3.75,7.75,7.75,2.8,2020,18,Bắc Giang


<class 'pandas.DataFrame'>
RangeIndex: 1857877 entries, 0 to 1857876
Data columns (total 16 columns):
 #   Column     Dtype  
---  ------     -----  
 0   SBD        int64  
 1   Tên        str    
 2   Ngày Sinh  str    
 3   Giới tính  str    
 4   Toán       float64
 5   Văn        float64
 6   Lý         float64
 7   Hoá        float64
 8   Sinh       float64
 9   Lịch Sử    float64
 10  Địa Lý     float64
 11  GDCD       float64
 12  Ngoại Ngữ  float64
 13  Year       int64  
 14  code       int64  
 15  province   str    
dtypes: float64(9), int64(3), str(4)
memory usage: 226.8 MB
None

--- 2022 ---


,sbd,toan,ngu_van,ngoai_ngu,vat_li,hoa_hoc,sinh_hoc,lich_su,dia_li,gdcd
0,1000001,3.6,5.00,4.0,NaN,NaN,NaN,2.75,6.0,8.75
1,1000002,8.4,6.75,7.6,NaN,NaN,NaN,8.50,7.5,8.25


<class 'pandas.DataFrame'>
RangeIndex: 995441 entries, 0 to 995440
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   sbd        995441 non-null  int64  
 1   toan       982726 non-null  float64
 2   ngu_van    981407 non-null  float64
 3   ngoai_ngu  870609 non-null  float64
 4   vat_li     325523 non-null  float64
 5   hoa_hoc    327367 non-null  float64
 6   sinh_hoc   322198 non-null  float64
 7   lich_su    659662 non-null  float64
 8   dia_li     657421 non-null  float64
 9   gdcd       554343 non-null  float64
dtypes: float64(9), int64(1)
memory usage: 75.9 MB
None

--- 2023 ---


...

<class 'pandas.DataFrame'>
RangeIndex: 22090 entries, 0 to 22089
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   STT                22090 non-null  int64  
 1   SOBAODANH          22090 non-null  int64  
 2   Toán               11245 non-null  float64
 3   Văn                17372 non-null  float64
 4   Lí                 4134 non-null   float64
 5   Hóa                4148 non-null   float64
 6   Sinh               1721 non-null   float64
 7   Sử                 13872 non-null  float64
 8   Địa                13109 non-null  float64
 9   Giáo dục công dân  4099 non-null   float64
 10  Ngoại ngữ          5252 non-null   float64
 11  Mã môn ngoại ngữ   5252 non-null   str    
dtypes: float64(9), int64(2), str(1)
memory usage: 2.0 MB
None

--- 2025_moi_1 ---


,STT,SOBAODANH,Toán,Văn,Lí,Hóa,Sinh,Tin học,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,Ngoại ngữ,Mã môn ngoại ngữ
0,1,1000001,5.75,7.75,NaN,7.75,8.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1000002,8.00,8.25,8.5,6.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 15 columns):
 #   Column                         Non-Null Count    Dtype  
---  ------                         --------------    -----  
 0   STT                            1000000 non-null  int64  
 1   SOBAODANH                      1000000 non-null  int64  
 2   Toán                           995699 non-null   float64
 3   Văn                            996142 non-null   float64
 4   Lí                             309882 non-null   float64
 5   Hóa                            209552 non-null   float64
 6   Sinh                           56730 non-null    float64
 7   Tin học                        6057 non-null     float64
 8   Công nghệ công nghiệp          1939 non-null     float64
 9   Công nghệ nông nghiệp          18231 non-null    float64
 10  Sử                             421604 non-null   float64
 11  Địa                            421004 non-null   float64
 12  Giáo dục kinh tế và pháp l

,STT,SOBAODANH,Toán,Văn,Lí,Hóa,Sinh,Tin học,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,Ngoại ngữ,Mã môn ngoại ngữ
0,1000001,52013349,3.25,6.75,5.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.5,N1
1,1000002,52013350,3.25,5.25,6.0,3.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 131136 entries, 0 to 131135
Data columns (total 15 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   STT                            131136 non-null  int64  
 1   SOBAODANH                      131136 non-null  int64  
 2   Toán                           130473 non-null  float64
 3   Văn                            130584 non-null  float64
 4   Lí                             37717 non-null   float64
 5   Hóa                            30583 non-null   float64
 6   Sinh                           13165 non-null   float64
 7   Tin học                        1545 non-null    float64
 8   Công nghệ công nghiệp          351 non-null     float64
 9   Công nghệ nông nghiệp          3817 non-null    float64
 10  Sử                             59689 non-null   float64
 11  Địa                            55468 non-null   float64
 12  Giáo dục kinh tế và pháp luật  31827 non-

In [99]:
# Visual inspection
for year, df in dfs.items():
    print(f"\n--- {year} ---")
    display(df.head(2))
    print(df.info())


--- 2021 ---


,SBD,Tên,Ngày Sinh,Giới tính,Toán,Văn,Lý,Hoá,Sinh,Lịch Sử,Địa Lý,GDCD,Ngoại Ngữ,Year,code,province
0,18014547,NaN,NaN,NaN,6.4,6.75,NaN,NaN,NaN,4.75,7.00,6.50,4.2,2020,18,Bắc Giang
1,18014530,NaN,NaN,NaN,7.6,6.00,NaN,NaN,NaN,3.75,7.75,7.75,2.8,2020,18,Bắc Giang


<class 'pandas.DataFrame'>
RangeIndex: 1857877 entries, 0 to 1857876
Data columns (total 16 columns):
 #   Column     Dtype  
---  ------     -----  
 0   SBD        int64  
 1   Tên        str    
 2   Ngày Sinh  str    
 3   Giới tính  str    
 4   Toán       float64
 5   Văn        float64
 6   Lý         float64
 7   Hoá        float64
 8   Sinh       float64
 9   Lịch Sử    float64
 10  Địa Lý     float64
 11  GDCD       float64
 12  Ngoại Ngữ  float64
 13  Year       int64  
 14  code       int64  
 15  province   str    
dtypes: float64(9), int64(3), str(4)
memory usage: 226.8 MB
None

--- 2022 ---


,sbd,toan,ngu_van,ngoai_ngu,vat_li,hoa_hoc,sinh_hoc,lich_su,dia_li,gdcd
0,1000001,3.6,5.00,4.0,NaN,NaN,NaN,2.75,6.0,8.75
1,1000002,8.4,6.75,7.6,NaN,NaN,NaN,8.50,7.5,8.25


<class 'pandas.DataFrame'>
RangeIndex: 995441 entries, 0 to 995440
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   sbd        995441 non-null  int64  
 1   toan       982726 non-null  float64
 2   ngu_van    981407 non-null  float64
 3   ngoai_ngu  870609 non-null  float64
 4   vat_li     325523 non-null  float64
 5   hoa_hoc    327367 non-null  float64
 6   sinh_hoc   322198 non-null  float64
 7   lich_su    659662 non-null  float64
 8   dia_li     657421 non-null  float64
 9   gdcd       554343 non-null  float64
dtypes: float64(9), int64(1)
memory usage: 75.9 MB
None

--- 2023 ---


...

<class 'pandas.DataFrame'>
RangeIndex: 22090 entries, 0 to 22089
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   STT                22090 non-null  int64  
 1   SOBAODANH          22090 non-null  int64  
 2   Toán               11245 non-null  float64
 3   Văn                17372 non-null  float64
 4   Lí                 4134 non-null   float64
 5   Hóa                4148 non-null   float64
 6   Sinh               1721 non-null   float64
 7   Sử                 13872 non-null  float64
 8   Địa                13109 non-null  float64
 9   Giáo dục công dân  4099 non-null   float64
 10  Ngoại ngữ          5252 non-null   float64
 11  Mã môn ngoại ngữ   5252 non-null   str    
dtypes: float64(9), int64(2), str(1)
memory usage: 2.0 MB
None

--- 2025_moi_1 ---


,STT,SOBAODANH,Toán,Văn,Lí,Hóa,Sinh,Tin học,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,Ngoại ngữ,Mã môn ngoại ngữ
0,1,1000001,5.75,7.75,NaN,7.75,8.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1000002,8.00,8.25,8.5,6.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 15 columns):
 #   Column                         Non-Null Count    Dtype  
---  ------                         --------------    -----  
 0   STT                            1000000 non-null  int64  
 1   SOBAODANH                      1000000 non-null  int64  
 2   Toán                           995699 non-null   float64
 3   Văn                            996142 non-null   float64
 4   Lí                             309882 non-null   float64
 5   Hóa                            209552 non-null   float64
 6   Sinh                           56730 non-null    float64
 7   Tin học                        6057 non-null     float64
 8   Công nghệ công nghiệp          1939 non-null     float64
 9   Công nghệ nông nghiệp          18231 non-null    float64
 10  Sử                             421604 non-null   float64
 11  Địa                            421004 non-null   float64
 12  Giáo dục kinh tế và pháp l

,STT,SOBAODANH,Toán,Văn,Lí,Hóa,Sinh,Tin học,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,Ngoại ngữ,Mã môn ngoại ngữ
0,1000001,52013349,3.25,6.75,5.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.5,N1
1,1000002,52013350,3.25,5.25,6.0,3.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 131136 entries, 0 to 131135
Data columns (total 15 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   STT                            131136 non-null  int64  
 1   SOBAODANH                      131136 non-null  int64  
 2   Toán                           130473 non-null  float64
 3   Văn                            130584 non-null  float64
 4   Lí                             37717 non-null   float64
 5   Hóa                            30583 non-null   float64
 6   Sinh                           13165 non-null   float64
 7   Tin học                        1545 non-null    float64
 8   Công nghệ công nghiệp          351 non-null     float64
 9   Công nghệ nông nghiệp          3817 non-null    float64
 10  Sử                             59689 non-null   float64
 11  Địa                            55468 non-null   float64
 12  Giáo dục kinh tế và pháp luật  31827 non-

Observations from assessing step:
1. **Data Types**: Scores are mostly numerical (float), but SBD/ID should be treated as strings to avoid losing leading zeros.
2. **Format**: Column names are inconsistent (uppercase/lowercase, English/Vietnamese, with/without underscores).
3. **Missing Values**: Many subjects have null values, which is expected as students don't take all elective subjects.
4. **Structural Consistency**: 
    - 2021-2024 follow a similar structure but with name variations.
    - 2024 introduces `ma_ngoai_ngu`.
    - 2025 is split into Old and New curricula; new curricula include subjects like Informatics and Technology.
5. **Quality**: Some encodings in 2021 headers are broken.

In [100]:
# Visual inspection
for year, df in dfs.items():
    print(f"\n--- {year} ---")
    display(df.head(2))
    print(df.info())


--- 2021 ---


,SBD,Tên,Ngày Sinh,Giới tính,Toán,Văn,Lý,Hoá,Sinh,Lịch Sử,Địa Lý,GDCD,Ngoại Ngữ,Year,code,province
0,18014547,NaN,NaN,NaN,6.4,6.75,NaN,NaN,NaN,4.75,7.00,6.50,4.2,2020,18,Bắc Giang
1,18014530,NaN,NaN,NaN,7.6,6.00,NaN,NaN,NaN,3.75,7.75,7.75,2.8,2020,18,Bắc Giang


<class 'pandas.DataFrame'>
RangeIndex: 1857877 entries, 0 to 1857876
Data columns (total 16 columns):
 #   Column     Dtype  
---  ------     -----  
 0   SBD        int64  
 1   Tên        str    
 2   Ngày Sinh  str    
 3   Giới tính  str    
 4   Toán       float64
 5   Văn        float64
 6   Lý         float64
 7   Hoá        float64
 8   Sinh       float64
 9   Lịch Sử    float64
 10  Địa Lý     float64
 11  GDCD       float64
 12  Ngoại Ngữ  float64
 13  Year       int64  
 14  code       int64  
 15  province   str    
dtypes: float64(9), int64(3), str(4)
memory usage: 226.8 MB
None

--- 2022 ---


,sbd,toan,ngu_van,ngoai_ngu,vat_li,hoa_hoc,sinh_hoc,lich_su,dia_li,gdcd
0,1000001,3.6,5.00,4.0,NaN,NaN,NaN,2.75,6.0,8.75
1,1000002,8.4,6.75,7.6,NaN,NaN,NaN,8.50,7.5,8.25


<class 'pandas.DataFrame'>
RangeIndex: 995441 entries, 0 to 995440
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   sbd        995441 non-null  int64  
 1   toan       982726 non-null  float64
 2   ngu_van    981407 non-null  float64
 3   ngoai_ngu  870609 non-null  float64
 4   vat_li     325523 non-null  float64
 5   hoa_hoc    327367 non-null  float64
 6   sinh_hoc   322198 non-null  float64
 7   lich_su    659662 non-null  float64
 8   dia_li     657421 non-null  float64
 9   gdcd       554343 non-null  float64
dtypes: float64(9), int64(1)
memory usage: 75.9 MB
None

--- 2023 ---


...

<class 'pandas.DataFrame'>
RangeIndex: 22090 entries, 0 to 22089
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   STT                22090 non-null  int64  
 1   SOBAODANH          22090 non-null  int64  
 2   Toán               11245 non-null  float64
 3   Văn                17372 non-null  float64
 4   Lí                 4134 non-null   float64
 5   Hóa                4148 non-null   float64
 6   Sinh               1721 non-null   float64
 7   Sử                 13872 non-null  float64
 8   Địa                13109 non-null  float64
 9   Giáo dục công dân  4099 non-null   float64
 10  Ngoại ngữ          5252 non-null   float64
 11  Mã môn ngoại ngữ   5252 non-null   str    
dtypes: float64(9), int64(2), str(1)
memory usage: 2.0 MB
None

--- 2025_moi_1 ---


,STT,SOBAODANH,Toán,Văn,Lí,Hóa,Sinh,Tin học,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,Ngoại ngữ,Mã môn ngoại ngữ
0,1,1000001,5.75,7.75,NaN,7.75,8.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1000002,8.00,8.25,8.5,6.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 15 columns):
 #   Column                         Non-Null Count    Dtype  
---  ------                         --------------    -----  
 0   STT                            1000000 non-null  int64  
 1   SOBAODANH                      1000000 non-null  int64  
 2   Toán                           995699 non-null   float64
 3   Văn                            996142 non-null   float64
 4   Lí                             309882 non-null   float64
 5   Hóa                            209552 non-null   float64
 6   Sinh                           56730 non-null    float64
 7   Tin học                        6057 non-null     float64
 8   Công nghệ công nghiệp          1939 non-null     float64
 9   Công nghệ nông nghiệp          18231 non-null    float64
 10  Sử                             421604 non-null   float64
 11  Địa                            421004 non-null   float64
 12  Giáo dục kinh tế và pháp l

,STT,SOBAODANH,Toán,Văn,Lí,Hóa,Sinh,Tin học,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,Ngoại ngữ,Mã môn ngoại ngữ
0,1000001,52013349,3.25,6.75,5.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.5,N1
1,1000002,52013350,3.25,5.25,6.0,3.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 131136 entries, 0 to 131135
Data columns (total 15 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   STT                            131136 non-null  int64  
 1   SOBAODANH                      131136 non-null  int64  
 2   Toán                           130473 non-null  float64
 3   Văn                            130584 non-null  float64
 4   Lí                             37717 non-null   float64
 5   Hóa                            30583 non-null   float64
 6   Sinh                           13165 non-null   float64
 7   Tin học                        1545 non-null    float64
 8   Công nghệ công nghiệp          351 non-null     float64
 9   Công nghệ nông nghiệp          3817 non-null    float64
 10  Sử                             59689 non-null   float64
 11  Địa                            55468 non-null   float64
 12  Giáo dục kinh tế và pháp luật  31827 non-

In [101]:
# Visual inspection
for year, df in dfs.items():
    print(f"\n--- {year} ---")
    display(df.head(2))
    print(df.info())


--- 2021 ---


,SBD,Tên,Ngày Sinh,Giới tính,Toán,Văn,Lý,Hoá,Sinh,Lịch Sử,Địa Lý,GDCD,Ngoại Ngữ,Year,code,province
0,18014547,NaN,NaN,NaN,6.4,6.75,NaN,NaN,NaN,4.75,7.00,6.50,4.2,2020,18,Bắc Giang
1,18014530,NaN,NaN,NaN,7.6,6.00,NaN,NaN,NaN,3.75,7.75,7.75,2.8,2020,18,Bắc Giang


<class 'pandas.DataFrame'>
RangeIndex: 1857877 entries, 0 to 1857876
Data columns (total 16 columns):
 #   Column     Dtype  
---  ------     -----  
 0   SBD        int64  
 1   Tên        str    
 2   Ngày Sinh  str    
 3   Giới tính  str    
 4   Toán       float64
 5   Văn        float64
 6   Lý         float64
 7   Hoá        float64
 8   Sinh       float64
 9   Lịch Sử    float64
 10  Địa Lý     float64
 11  GDCD       float64
 12  Ngoại Ngữ  float64
 13  Year       int64  
 14  code       int64  
 15  province   str    
dtypes: float64(9), int64(3), str(4)
memory usage: 226.8 MB
None

--- 2022 ---


,sbd,toan,ngu_van,ngoai_ngu,vat_li,hoa_hoc,sinh_hoc,lich_su,dia_li,gdcd
0,1000001,3.6,5.00,4.0,NaN,NaN,NaN,2.75,6.0,8.75
1,1000002,8.4,6.75,7.6,NaN,NaN,NaN,8.50,7.5,8.25


<class 'pandas.DataFrame'>
RangeIndex: 995441 entries, 0 to 995440
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   sbd        995441 non-null  int64  
 1   toan       982726 non-null  float64
 2   ngu_van    981407 non-null  float64
 3   ngoai_ngu  870609 non-null  float64
 4   vat_li     325523 non-null  float64
 5   hoa_hoc    327367 non-null  float64
 6   sinh_hoc   322198 non-null  float64
 7   lich_su    659662 non-null  float64
 8   dia_li     657421 non-null  float64
 9   gdcd       554343 non-null  float64
dtypes: float64(9), int64(1)
memory usage: 75.9 MB
None

--- 2023 ---


...

<class 'pandas.DataFrame'>
RangeIndex: 22090 entries, 0 to 22089
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   STT                22090 non-null  int64  
 1   SOBAODANH          22090 non-null  int64  
 2   Toán               11245 non-null  float64
 3   Văn                17372 non-null  float64
 4   Lí                 4134 non-null   float64
 5   Hóa                4148 non-null   float64
 6   Sinh               1721 non-null   float64
 7   Sử                 13872 non-null  float64
 8   Địa                13109 non-null  float64
 9   Giáo dục công dân  4099 non-null   float64
 10  Ngoại ngữ          5252 non-null   float64
 11  Mã môn ngoại ngữ   5252 non-null   str    
dtypes: float64(9), int64(2), str(1)
memory usage: 2.0 MB
None

--- 2025_moi_1 ---


,STT,SOBAODANH,Toán,Văn,Lí,Hóa,Sinh,Tin học,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,Ngoại ngữ,Mã môn ngoại ngữ
0,1,1000001,5.75,7.75,NaN,7.75,8.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1000002,8.00,8.25,8.5,6.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 15 columns):
 #   Column                         Non-Null Count    Dtype  
---  ------                         --------------    -----  
 0   STT                            1000000 non-null  int64  
 1   SOBAODANH                      1000000 non-null  int64  
 2   Toán                           995699 non-null   float64
 3   Văn                            996142 non-null   float64
 4   Lí                             309882 non-null   float64
 5   Hóa                            209552 non-null   float64
 6   Sinh                           56730 non-null    float64
 7   Tin học                        6057 non-null     float64
 8   Công nghệ công nghiệp          1939 non-null     float64
 9   Công nghệ nông nghiệp          18231 non-null    float64
 10  Sử                             421604 non-null   float64
 11  Địa                            421004 non-null   float64
 12  Giáo dục kinh tế và pháp l

,STT,SOBAODANH,Toán,Văn,Lí,Hóa,Sinh,Tin học,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,Ngoại ngữ,Mã môn ngoại ngữ
0,1000001,52013349,3.25,6.75,5.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.5,N1
1,1000002,52013350,3.25,5.25,6.0,3.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 131136 entries, 0 to 131135
Data columns (total 15 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   STT                            131136 non-null  int64  
 1   SOBAODANH                      131136 non-null  int64  
 2   Toán                           130473 non-null  float64
 3   Văn                            130584 non-null  float64
 4   Lí                             37717 non-null   float64
 5   Hóa                            30583 non-null   float64
 6   Sinh                           13165 non-null   float64
 7   Tin học                        1545 non-null    float64
 8   Công nghệ công nghiệp          351 non-null     float64
 9   Công nghệ nông nghiệp          3817 non-null    float64
 10  Sử                             59689 non-null   float64
 11  Địa                            55468 non-null   float64
 12  Giáo dục kinh tế và pháp luật  31827 non-

## 3. Clean data
Clean the data to solve the 4 issues corresponding to data quality and tidiness found in the assessing step. 


### Step 0: Make copies of the datasets

In [102]:
# Make copies of the datasets to ensure the raw dataframes are not impacted
clean_dfs = {year: df.copy() for year, df in dfs.items()}

### **Issue 1: Inconsistent Column Names and Encoding Issues**

#### **Define:**
Normalize and map column headers to a common subject naming convention across all years to resolve inconsistencies caused by language differences and varied encoding (UTF-8 vs Latin1).

In [103]:
def standardize_headers(df):
    import unicodedata
    import re
    
    def clean_name(name):
        name = str(name).lower().strip().replace('đ', 'd')
        name = ''.join(c for c in unicodedata.normalize('NFD', name) if unicodedata.category(c) != 'Mn')
        name = re.sub(r'[^a-z0-9]', '_', name)
        return re.sub(r'_+', '_', name).strip('_')

    # Initial cleaning
    df.columns = [clean_name(c) for c in df.columns]
    
    mapping = {
        'toan': 'toan', 'van': 'ngu_van', 'ngu_van': 'ngu_van', 
        'li': 'vat_li', 'vat_li': 'vat_li', 'vat_ly': 'vat_li', 
        'hoa': 'hoa_hoc', 'hoa_hoc': 'hoa_hoc', 
        'sinh': 'sinh_hoc', 'sinh_hoc': 'sinh_hoc', 
        'su': 'lich_su', 'lich_su': 'lich_su', 
        'ia': 'dia_li', 'dia_li': 'dia_li', 'dia': 'dia_li', 
        'gdcd': 'gdcd', 'giao_duc_cong_dan': 'gdcd', 
        'tin_hoc': 'tin_hoc', 
        'giao_duc_kinh_te_va_phap_luat': 'gd_ktpl', 'gd_ktpl': 'gd_ktpl', 'ktpl': 'gd_ktpl', 
        'ngoai_ngu': 'ngoai_ngu', 'sobaodanh': 'sbd', 'sbd': 'sbd'
    }
    
    # Create unique mapping
    new_names = []
    seen = set()
    for c in df.columns:
        target = c
        # Try exact match or substring match
        for key, val in mapping.items():
            if key == c or key in c:
                target = val
                break
        
        # Enforce uniqueness
        final_name = target
        counter = 1
        while final_name in seen:
            final_name = f"{target}_{counter}"
            counter += 1
        seen.add(final_name)
        new_names.append(final_name)
    
    df.columns = new_names
    return df

for key in clean_dfs:
    clean_dfs[key] = standardize_headers(clean_dfs[key])

#### **Justification:**
Ensures that all year-specific subject variations are unified under a single schema, allowing for programmatic comparison throughout the analysis.

### **Issue 2: Incorrect Data Types for Student ID (SBD)**

#### **Define:**
Convert 'sbd' columns to string objects and pad them to a standard length (typically 8 digits) to restore leading zeros lost during ingestion.

In [104]:
for key, df in clean_dfs.items():
    if 'sbd' in df.columns:
        # Convert to string and pad with zeros to 8 digits
        df['sbd'] = df['sbd'].astype(str).str.zfill(8)

#### **Justification:**
Leading zeros in Vietnam's SBD system represent the province/city code. Correcting this data type is critical for identifying student locations properly.

### **Issue 3: Structural Differences between Old and New Curricula**

#### **Define:**
Label each dataset with its exam 'year' and 'program' (old vs new) before merging them into a single master CSV-ready dataframe.

In [105]:
all_cleaned = []
for key, df in clean_dfs.items():
    df['year'] = key.split('_')[0]
    df['program'] = 'new' if 'moi' in key else 'old'
    all_cleaned.append(df)

master_df = pd.concat(all_cleaned, ignore_index=True)

#### **Justification:**
This adds the necessary dimension for comparative research questions while using Pandas' ability to handle sparse subjects (NaN) for students who only took specific elective exams.

### **Issue 4: Presence of Irrelevant Metadata (STT, Name, Province)**

#### **Define:**
Filter the master dataframe to retain only required columns and convert all score entries to numeric types, coercing errors to NaN.

In [106]:
cols_to_keep = ['sbd', 'toan', 'ngu_van', 'ngoai_ngu', 'vat_li', 'hoa_hoc', 'sinh_hoc', 
                'lich_su', 'dia_li', 'gdcd', 'gd_ktpl', 'tin_hoc', 'year', 'program']
master_df = master_df[[c for c in cols_to_keep if c in master_df.columns]]

score_cols = ['toan', 'ngu_van', 'ngoai_ngu', 'vat_li', 'hoa_hoc', 'sinh_hoc', 
              'lich_su', 'dia_li', 'gdcd', 'gd_ktpl', 'tin_hoc']
for col in score_cols:
    if col in master_df.columns:
        master_df[col] = pd.to_numeric(master_df[col], errors='coerce')

print(f"Master Dataframe shape: {master_df.shape}")

Master Dataframe shape: (6090209, 13)


#### **Justification:**
Cleaning the column set and data types ensures a reliable, anonymized dataset optimized for high-performance statistical analysis.

## 4. Update your data store
Update the database/data store with the cleaned data


In [107]:
master_df.to_csv('dataset/cleaned_all_years.csv', index=False)
print('Data saved successfully!')

Data saved successfully!


## 5. Answer the research question

### **5.1:** Define and answer the research question 


* Question 1: Tốc độ tăng trưởng số lượng thí sinh dự thi năm 2025 so với giai đoạn 2020-2024 thay đổi ra sao, và sự gia tăng khối lượng thí sinh này tạo ra áp lực cạnh tranh (tỷ lệ chọi) như thế nào lên các nhóm trường Đại học top đầu?

In [108]:
#Visual 1 - FILL IN

Justifications

* Question 2: Có sự dịch chuyển bất thường nào trong phổ điểm của các môn học từng có hiện tượng "tăng vọt" về điểm số trong năm 2024 (như Lịch sử, Địa lý, Ngữ Văn) hay không? Đề thi năm 2025 đã đưa phổ điểm các môn này về mức cân bằng hay tiếp tục lạm phát điểm?

In [109]:
#Visual 2 - FILL IN

Justifications

* Question 3: Ngưỡng điểm để lọt vào "Top 5% và Top 10% thí sinh xuất sắc nhất" ở các khối thi truyền thống (A, A1, B, C, D) năm 2025 dịch chuyển bao nhiêu điểm so với 2024?

In [110]:
#Visual 3 - FILL IN

Justifications

* Question 4: Tại các vùng điểm nóng cạnh tranh (từ 24 đến 28 điểm), số lượng thí sinh tích lũy dư thừa hay thiếu hụt bao nhiêu người so với năm 2024 ở từng khối thi; kết hợp với biến số về chỉ tiêu và phương thức xét tuyển, điểm chuẩn của các trường Đại học top đầu (như Bách Khoa, Ngoại Thương, Kinh Tế Quốc Dân,...) sẽ biến động theo xu hướng nào?

In [111]:
#Visual 4 - FILL IN

Justifications

* Question 5: Mối tương quan điểm số giữa các môn học (Ví dụ: Toán và Ngoại ngữ, hoặc Ngữ Văn và Lịch Sử) năm 2025 có sự thay đổi nào không? Những thí sinh học giỏi môn Tự nhiên có xu hướng đạt điểm cao môn Ngoại ngữ như các năm trước hay không?

In [112]:
#Visual 5 - FILL IN

Justifications

* Question 6: Bản đồ phân bổ trung vị điểm và "độ lệch chuẩn" năm 2025 cho thấy khoảng cách về năng lực học tập và chất lượng giáo dục giữa các thành phố lớn (Hà Nội, TP.HCM) với các khu vực miền núi/vùng ven đang thu hẹp hay ngày càng giãn rộng?

In [113]:
#Visual 6 - FILL IN

Justifications

In [114]:
#Visual 1 - FILL IN


### **5.2:** Reflection

After all steps above, I have clean datasets and answers for the research questions.
By using research questions, we can understand the score distributions across different years, identify inflation or deflation in subject grades, and project university entrance requirements.

In the future, I will look into more granular regional data and the impact of different admission methods on the effective benchmark scores.
Furthermore, I will analyze the correlation between specific subject pairs to identify emerging student strengths.

#### SUBMISSION

After completion, export the notebook as an HTML file for the project submission using the File > Download as... > HTML or PDF menu option.

Submit file “my_Project.zip” to Final Project includes:

    1. my_Project.html
    2. my _Project.ipynb
